# Creación de Variable Objetivo: target_derm

Este notebook documenta el proceso de creación de la variable objetivo binaria `target_derm` para la predicción de enfermedades veterinarias.

**Objetivo:** Clasificar casos como dermatológicos (1) o no dermatológicos (0) basándose en síntomas e historial médico.

## Importar Librerías y Cargar Dataset

In [1]:
import pandas as pd
import numpy as np
import re

# Cargar dataset limpio
df = pd.read_csv('../veterinary_clinical_dataset_clean.csv')

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\nPrimeras filas:")
df.head()

Dataset cargado: 5013 filas, 10 columnas

Primeras filas:


,AnimalName,Breed,Age,Weight_kg,MedicalHistory,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5
0,dog,rottweiler,6.0,32.1,chronic illness,anorexia,hydrophobia,drooping ears,diarrhea,shyness or aggression
1,dog,bulldog,9.9,18.5,vaccinated,lethargy,weakness,horny growth,fever,coughing
2,dog,beagle,13.9,18.9,parasite history,pain,weight loss,weight loss,sneezing,drop on egg production
3,dog,mixed breed,4.2,13.4,chronic illness,weakness,diarrhea,weight loss,diarrhea,weight loss
4,dog,boxer,2.6,27.6,dental issues,indigestion,abdminal pain,constipation,diarrhea,nausea


## Normalización de Texto

Aplicamos normalización a todas las columnas de texto (síntomas e historial médico) para:
- Convertir a minúsculas
- Eliminar espacios extra
- Corregir typos comunes identificados en el EDA

In [2]:
def norm_txt(s):
    """Normaliza texto: lowercase, trim, corrección de typos."""
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)  # Eliminar espacios múltiples
    
    # Corrección de typos comunes
    s = s.replace('abdminal', 'abdominal')
    s = s.replace('twiching', 'twitching')
    s = s.replace('fell unwell', 'felt unwell')
    s = s.replace('drop on egg production', 'drop in egg production')
    
    return s

# Identificar columnas de texto
text_cols = [c for c in df.columns if any(k in c.lower() for k in ['medicalhistory', 'symptom'])]
print(f"Columnas de texto identificadas: {text_cols}")

# Aplicar normalización
for c in text_cols:
    df[c + '_norm'] = df[c].apply(norm_txt)

print("\nEjemplo de normalización:")
print(df[['Symptom_1', 'Symptom_1_norm']].head(10))

Columnas de texto identificadas: ['MedicalHistory', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5']

Ejemplo de normalización:
                   Symptom_1             Symptom_1_norm
0                   anorexia                   anorexia
1                   lethargy                   lethargy
2                       pain                       pain
3                   weakness                   weakness
4                indigestion                indigestion
5                   weakness                   weakness
6                      fever                      fever
7                   weakness                   weakness
8                   vomiting                   vomiting
9  abortion on late pregancy  abortion on late pregancy


## Definición de Keywords Dermatológicas

Definimos dos conjuntos de keywords:
1. **Síntomas dermatológicos:** palabras clave que indican condiciones de piel
2. **Historial dermatológico:** patrones en el historial médico relacionados con piel

In [3]:
# Keywords para síntomas dermatológicos
symptom_derm_keywords = [
    'dermatitis', 'ringworm', 'fungal', 'fungal_infections', 'hypersensitivity', 
    'demodicosis', 'lesion', 'rash', 'alopecia', 'pruritus', 'itch', 'hot spot', 
    'hot_spot', 'skin', 'scabies', 'mange', 'pyoderma', 'erythema', 'papule', 
    'pustule', 'crust', 'ulcer', 'scab', 'otitis'
]

# Keywords contextuales para 'swelling' (solo derm si aparece con skin/ear)
context_keywords = ['skin', 'ear', 'otitis']

# Keywords para historial médico
hist_derm_keywords = [
    'skin conditions history', 'skin condition', 'dermatitis', 'ringworm',
    'fungal', 'hypersensitivity', 'demodicosis', 'otitis'
]

print(f"Total de keywords dermatológicas en síntomas: {len(symptom_derm_keywords)}")
print(f"Total de keywords dermatológicas en historial: {len(hist_derm_keywords)}")

Total de keywords dermatológicas en síntomas: 24
Total de keywords dermatológicas en historial: 8


## Función de Detección Dermatológica

Creamos una función que determina si un texto de síntoma contiene indicadores dermatológicos.

In [5]:
def is_derm_symptom(text: str) -> bool:
    """Detecta si un texto de síntoma es dermatológico."""
    if not text:
        return False
    
    # Búsqueda directa de keywords
    for kw in symptom_derm_keywords:
        if kw in text:
            return True
    
    # Swelling/swollen solo si aparece con contexto de piel/oído
    if 'swelling' in text or 'swollen' in text:
        for ck in context_keywords:
            if ck in text:
                return True
    
    return False

# Test de la función
test_cases = [
    'dermatitis',
    'skin lesion',
    'swelling of joints',  # No derm
    'swelling of skin',    # Derm
    'fever'
]

print("Test de función is_derm_symptom:")
for tc in test_cases:
    print(f"  '{tc}' -> {is_derm_symptom(tc)}")

Test de función is_derm_symptom:
  'dermatitis' -> True
  'skin lesion' -> True
  'swelling of joints' -> False
  'swelling of skin' -> True
  'fever' -> False


## Creación de la Variable Objetivo

**Regla de etiquetado:**
- `target_derm = 1` si:
  - Cualquier síntoma (Symptom_1 a Symptom_5) contiene una keyword dermatológica, O
  - El historial médico contiene una keyword dermatológica
- `target_derm = 0` en caso contrario

In [6]:
# Identificar columnas de síntomas normalizadas
symptom_cols_norm = [c for c in df.columns if c.lower().startswith('symptom_') and c.endswith('_norm')]
print(f"Columnas de síntomas normalizadas: {symptom_cols_norm}")

# Flag dermatológico desde síntomas
symptom_derm_flag = np.zeros(len(df), dtype=bool)
for c in symptom_cols_norm:
    symptom_derm_flag = symptom_derm_flag | df[c].apply(is_derm_symptom).values

print(f"\nCasos dermatológicos detectados por síntomas: {symptom_derm_flag.sum()}")

# Flag dermatológico desde historial médico
hist_col = [c for c in df.columns if c.lower().startswith('medicalhistory') and c.endswith('_norm')]
if hist_col:
    hcol = hist_col[0]
    hist_derm_flag = df[hcol].fillna('').apply(
        lambda t: any(kw in t for kw in hist_derm_keywords)
    ).values
else:
    hist_derm_flag = np.zeros(len(df), dtype=bool)

print(f"Casos dermatológicos detectados por historial: {hist_derm_flag.sum()}")

# Variable objetivo final
df['target_derm'] = (symptom_derm_flag | hist_derm_flag).astype(int)

print(f"\nTotal de casos dermatológicos (target_derm=1): {df['target_derm'].sum()}")

Columnas de síntomas normalizadas: ['Symptom_1_norm', 'Symptom_2_norm', 'Symptom_3_norm', 'Symptom_4_norm', 'Symptom_5_norm']

Casos dermatológicos detectados por síntomas: 568
Casos dermatológicos detectados por historial: 534

Total de casos dermatológicos (target_derm=1): 1020


## Análisis de Balance de Clases

In [7]:
# Balance de clases
balance = df['target_derm'].value_counts(normalize=True).rename('proportion')
counts = df['target_derm'].value_counts().rename('count')
summary = pd.concat([counts, balance], axis=1)

print("Balance de clases:")
print(summary)
print(f"\nRatio No-Derm:Derm = {summary.loc[0, 'count']/summary.loc[1, 'count']:.2f}:1")

Balance de clases:
             count  proportion
target_derm                   
0             3993    0.796529
1             1020    0.203471

Ratio No-Derm:Derm = 3.91:1


## Validación Manual (Muestra)

Revisamos algunos casos para validar la calidad del etiquetado.

In [8]:
# Muestra de casos dermatológicos
print("=== MUESTRA DE CASOS DERMATOLÓGICOS (target_derm=1) ===")
derm_sample = df[df['target_derm']==1][['Breed', 'Age', 'MedicalHistory', 
                                         'Symptom_1', 'Symptom_2', 'Symptom_3', 
                                         'target_derm']].head(10)
print(derm_sample.to_string())

print("\n=== MUESTRA DE CASOS NO DERMATOLÓGICOS (target_derm=0) ===")
non_derm_sample = df[df['target_derm']==0][['Breed', 'Age', 'MedicalHistory', 
                                             'Symptom_1', 'Symptom_2', 'Symptom_3', 
                                             'target_derm']].head(10)
print(non_derm_sample.to_string())

=== MUESTRA DE CASOS DERMATOLÓGICOS (target_derm=1) ===
                Breed   Age            MedicalHistory    Symptom_1               Symptom_2                                   Symptom_3  target_derm
5   yorkshire terrier  12.4  previous heart condition     weakness             weight loss                             muscle twiching            1
7    golden retriever  11.4   skin conditions history     weakness                    pain                                    vomiting            1
22        mixed breed   4.5   skin conditions history    enteritis             weight loss                                   halitosis            1
25         rottweiler   4.1   skin conditions history     sneezing                coughing                                       fever            1
29             beagle   9.2            recent surgery      redness                   pains                                     trachea            1
32            bulldog   8.5   skin conditions history   

## Guardar Dataset con Variable Objetivo

In [9]:
# Guardar dataset con target
output_file = 'veterinary_clinical_dataset_with_target.csv'
df.to_csv(output_file, index=False)

print(f"Dataset guardado exitosamente: {output_file}")
print(f"Dimensiones finales: {df.shape}")
print(f"\nColumnas añadidas:")
new_cols = [c for c in df.columns if c not in ['AnimalName', 'Breed', 'Age', 'Weight_kg', 
                                                 'MedicalHistory', 'Symptom_1', 'Symptom_2', 
                                                 'Symptom_3', 'Symptom_4', 'Symptom_5']]
for col in new_cols:
    print(f"  - {col}")

Dataset guardado exitosamente: veterinary_clinical_dataset_with_target.csv
Dimensiones finales: (5013, 17)

Columnas añadidas:
  - MedicalHistory_norm
  - Symptom_1_norm
  - Symptom_2_norm
  - Symptom_3_norm
  - Symptom_4_norm
  - Symptom_5_norm
  - target_derm


## 9. Conclusiones

- Se creó exitosamente la variable objetivo binaria `target_derm`
- Balance de clases: ~80% no dermatológico, ~20% dermatológico
- El desbalance es manejable con técnicas estándar (class_weight, SMOTE, threshold tuning)
- Se recomienda validar manualmente una muestra más amplia de casos borderline
- Próximo paso: Feature Engineering sobre este dataset